In [1]:
from pinecone import Pinecone, ServerlessSpec
import os
from openai import OpenAI
import pandas as pd
from time import time
import dotenv
dotenv.load_dotenv()


from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import DirectoryLoader, PyPDFLoader, UnstructuredPowerPointLoader


c:\Users\Hp\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
token= os.getenv("RUNPOD_TOKEN") 
open_ai_base_url = os.getenv("RUNPOD_EMBEDDING_URL") 
model_name= os.getenv("MODEL_NAME") 
pinecone_api_key = os.getenv("PINECONE_API_KEY") 

In [3]:
pc = Pinecone(api_key=pinecone_api_key)

client = OpenAI(
  api_key=token, 
  base_url=open_ai_base_url
)

## Try out embeddings

In [4]:
output = client.embeddings.create(input = ["helloo there"],model=model_name)
embedings = output.data[0].embedding
print(embedings)

[-0.05535165220499039, -0.056572869420051575, 0.08585154265165329, -0.06237364932894707, 0.01749393157660961, -0.011418377049267292, 0.052420731633901596, 0.052603915333747864, 0.02895810268819332, -0.02263830602169037, -0.013250201940536499, -0.04787169769406319, 0.029446590691804886, 0.031751636415719986, 0.05635915696620941, -0.009006474167108536, 0.012746450491249561, -0.054222024977207184, -0.103620246052742, -0.019554734230041504, 0.029156550765037537, 0.05199330672621727, -0.028805451467633247, -0.034804679453372955, 0.0015770869795233011, -0.006640366278588772, 0.019661590456962585, 0.03562900051474571, -0.008487456478178501, -0.07376149296760559, -0.00048514746595174074, -0.012944898568093777, 0.04658942297101021, -0.0003446598129812628, 0.051382698118686676, 0.0005123385926708579, 0.059625908732414246, -0.025065474212169647, -0.07791363447904587, -0.005609964486211538, 0.0604502335190773, -0.025019679218530655, -0.0016944382805377245, -0.016669608652591705, 0.0225619804114103

In [5]:
len(embedings)

384

## Wrangle dataset

In [4]:
df=pd.read_json('data/students.json')

In [5]:
df.head(2)

,full_name,registration_number,email,region,description,image_path,year,department,status
0,Hiba Mansour,S12255237,s12255237@stu.najah.edu,Palestine,Computer science student with a passion for da...,hiba_mansour.jpg,3rd Year,Computer Science,active
1,Mai Shelbayeh,N/A,Maishelbayeh@icloud.com,Palestine,AI enthusiast with interest in mobile app deve...,mai_shelbayeh.jpg,1st Year,Software Engineering,active


In [6]:
intro = "The following students are currently enrolled in the Smart System Management course.\n"

df['text'] = df['full_name'] + " : " + df['description'] + \
             " -- Registration Number: " + df['registration_number'] + \
             " -- Email: " + df['email'] + \
             " -- Department: " + df['department'] + \
             " -- Year: " + df['year'] + \
             " -- Status: " + df['status']

# Combine all rows into a single string
full_text = intro + "\n".join(df['text'])

print(full_text)


The following students are currently enrolled in the Smart System Management course.
Hiba Mansour : Computer science student with a passion for data analysis. -- Registration Number: S12255237 -- Email: s12255237@stu.najah.edu -- Department: Computer Science -- Year: 3rd Year -- Status: active
Mai Shelbayeh : AI enthusiast with interest in mobile app development. -- Registration Number: N/A -- Email: Maishelbayeh@icloud.com -- Department: Software Engineering -- Year: 1st Year -- Status: active
Orwa Jabali : Interested in smart systems and human-computer interaction. -- Registration Number: s12255138 -- Email: Orwajabali89@gmail.com -- Department: Artificial Intelligence -- Year: 3rd Year -- Status: active
Raghad Hethnawi : Aspiring researcher in natural language processing. -- Registration Number: S12457356 -- Email: s12457356@stu.najah.edu -- Department: Computer Science -- Year: 1st Year -- Status: active
Aseel Omar : Focused on cybersecurity and network systems. -- Registration Num

In [9]:
# Show full text without truncation
pd.set_option('display.max_colwidth', None)

# Show the first row's text column
print(df['text'].head())

0          Hiba Mansour : Computer science student with a passion for data analysis. -- Registration Number: S12255237 -- Email: s12255237@stu.najah.edu -- Department: Computer Science -- Year: 3rd Year -- Status: active
1               Mai Shelbayeh : AI enthusiast with interest in mobile app development. -- Registration Number: N/A -- Email: Maishelbayeh@icloud.com -- Department: Software Engineering -- Year: 1st Year -- Status: active
2    Orwa Jabali : Interested in smart systems and human-computer interaction. -- Registration Number: s12255138 -- Email: Orwajabali89@gmail.com -- Department: Artificial Intelligence -- Year: 3rd Year -- Status: active
3              Raghad Hethnawi : Aspiring researcher in natural language processing. -- Registration Number: S12457356 -- Email: s12457356@stu.najah.edu -- Department: Computer Science -- Year: 1st Year -- Status: active
4                     Aseel Omar : Focused on cybersecurity and network systems. -- Registration Number: S12356791 -

In [12]:
texts = df['text'].tolist()

In [13]:
texts

['Hiba Mansour : Computer science student with a passion for data analysis. -- Registration Number: S12255237 -- Email: s12255237@stu.najah.edu -- Department: Computer Science -- Year: 3rd Year -- Status: active',
 'Mai Shelbayeh : AI enthusiast with interest in mobile app development. -- Registration Number: N/A -- Email: Maishelbayeh@icloud.com -- Department: Software Engineering -- Year: 1st Year -- Status: active',
 'Orwa Jabali : Interested in smart systems and human-computer interaction. -- Registration Number: s12255138 -- Email: Orwajabali89@gmail.com -- Department: Artificial Intelligence -- Year: 3rd Year -- Status: active',
 'Raghad Hethnawi : Aspiring researcher in natural language processing. -- Registration Number: S12457356 -- Email: s12457356@stu.najah.edu -- Department: Computer Science -- Year: 1st Year -- Status: active',
 'Aseel Omar : Focused on cybersecurity and network systems. -- Registration Number: S12356791 -- Email: s12356791@stu.najah.edu -- Department: Inf

In [14]:
texts

['Hiba Mansour : Computer science student with a passion for data analysis. -- Registration Number: S12255237 -- Email: s12255237@stu.najah.edu -- Department: Computer Science -- Year: 3rd Year -- Status: active',
 'Mai Shelbayeh : AI enthusiast with interest in mobile app development. -- Registration Number: N/A -- Email: Maishelbayeh@icloud.com -- Department: Software Engineering -- Year: 1st Year -- Status: active',
 'Orwa Jabali : Interested in smart systems and human-computer interaction. -- Registration Number: s12255138 -- Email: Orwajabali89@gmail.com -- Department: Artificial Intelligence -- Year: 3rd Year -- Status: active',
 'Raghad Hethnawi : Aspiring researcher in natural language processing. -- Registration Number: S12457356 -- Email: s12457356@stu.najah.edu -- Department: Computer Science -- Year: 1st Year -- Status: active',
 'Aseel Omar : Focused on cybersecurity and network systems. -- Registration Number: S12356791 -- Email: s12356791@stu.najah.edu -- Department: Inf

In [15]:
with open('data/Prof_Allam_Mousa_CV.txt') as f:
    Prof_Allam_Mousa_CV = f.read()
    
Prof_Allam_Mousa_CV = "Professor Allam Mousa - Curriculum Vitae: " + Prof_Allam_Mousa_CV
texts.append(Prof_Allam_Mousa_CV)

In [16]:
with open('data/SSM_Syllabus.txt') as f:
    SSM_Syllabus = f.read()
    
SSM_Syllabus = "Course Syllabus: " + SSM_Syllabus
texts.append(SSM_Syllabus)

In [17]:
texts

['Hiba Mansour : Computer science student with a passion for data analysis. -- Registration Number: S12255237 -- Email: s12255237@stu.najah.edu -- Department: Computer Science -- Year: 3rd Year -- Status: active',
 'Mai Shelbayeh : AI enthusiast with interest in mobile app development. -- Registration Number: N/A -- Email: Maishelbayeh@icloud.com -- Department: Software Engineering -- Year: 1st Year -- Status: active',
 'Orwa Jabali : Interested in smart systems and human-computer interaction. -- Registration Number: s12255138 -- Email: Orwajabali89@gmail.com -- Department: Artificial Intelligence -- Year: 3rd Year -- Status: active',
 'Raghad Hethnawi : Aspiring researcher in natural language processing. -- Registration Number: S12457356 -- Email: s12457356@stu.najah.edu -- Department: Computer Science -- Year: 1st Year -- Status: active',
 'Aseel Omar : Focused on cybersecurity and network systems. -- Registration Number: S12356791 -- Email: s12356791@stu.najah.edu -- Department: Inf

## Generate Embeddings

In [18]:
output = client.embeddings.create(input = texts,model=model_name)

In [19]:
embeddings = output.data

In [20]:
embeddings

[Embedding(embedding=[-0.007055779919028282, 0.04588355869054794, -0.05385136231780052, -0.06532987952232361, 0.01749253459274769, 0.010410042479634285, 0.04948585852980614, 0.03355025127530098, -0.025704560801386833, -0.05705679953098297, 0.023689715191721916, -0.07540411502122879, 0.06484143435955048, -0.014401575550436974, 0.07583151012659073, 0.00792201142758131, -0.03605354577302933, 0.014638167805969715, 0.047867875546216965, -0.05073750764131546, -0.02978004701435566, -0.028742095455527306, -0.060445405542850494, -0.036572523415088654, 0.03751889243721962, 0.042525481432676315, 0.012989656999707222, -0.05748419091105461, -0.10239085555076599, -0.15532638132572174, 0.02920001558959484, 0.02721569687128067, 0.047287844121456146, 0.016820918768644333, 0.03028375841677189, -0.023002834990620613, -0.004583013243973255, -0.010387145914137363, -0.02457502670586109, -0.025765616446733475, -0.03788522630929947, -0.04389923810958862, 0.03605354577302933, 0.015302151441574097, 0.0339471176

## Push data to database

### Initialize Pinecone and Create Index (This is Knowledge Base)

In [21]:
index_name = "smart-system-management-chatbot"

# pc.create_index(
#     name=index_name,
#     dimension=384, # Replace with your model dimensions
#     metric="cosine", # Replace with your model metric
#     spec=ServerlessSpec(
#         cloud="aws",
#         region="us-east-1"
#     ) 
# )

In [22]:
# Wait for the index to be ready
while not pc.describe_index(index_name).status['ready']:
    time.sleep(1)

index = pc.Index(index_name)

vectors = []
for text, e in zip(texts, embeddings):
    entry_id = text.split(":")[0].strip()
    vectors.append({
        "id": entry_id,
        "values": e.embedding,
        "metadata": {'text': text}
    })
    
index.upsert(
    vectors=vectors,
    namespace="Students_Instructors_Info"
)

{'upserted_count': 17}

## Get Closest documents

In [48]:
output = client.embeddings.create(input = ["name 2 of students that an Artificial Intelligence engineer"],model=model_name)
embeding = output.data[0].embedding

In [49]:
results = index.query(
    namespace="Students_Instructors_Info",
    vector=embeding,
    top_k=3,
    include_values=False,
    include_metadata=True
)

print(results)

{'matches': [{'id': 'The following students are currently enrolled in the '
                    'Smart System Management course.\n'
                    'Hiba Mansour',
              'metadata': {'text': 'The following students are currently '
                                   'enrolled in the Smart System Management '
                                   'course.\n'
                                   'Hiba Mansour : Computer science student '
                                   'with a passion for data analysis. -- '
                                   'Registration Number: S12255237 -- Email: '
                                   's12255237@stu.najah.edu -- Department: '
                                   'Computer Science -- Year: 3rd Year -- '
                                   'Status: active\n'
                                   'Mai Shelbayeh : AI enthusiast with '
                                   'interest in mobile app development. -- '
                                   'Registr

In [51]:
# Step 1: Extract metadata (texts) from the query results
# Only use top 2 and truncate context if needed
retrieved_contexts = [match['metadata']['text'][:1000] for match in results['matches'][:2]]  # first 1000 characters max


# Step 2: Prepare the prompt
query = "Name 2 students that are Artificial Intelligence engineers."

context = "\n".join(retrieved_contexts)
final_prompt = f"""Use the following information to answer the question.

Context:
{context}

Question:
{query}

Answer:"""

# Step 3: Call the LLM using RunPod Chat Completion
chat_response = client.chat.completions.create(
    model=model_name,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": final_prompt}
    ],
    temperature=0.2
)

# Step 4: Print the answer
print(chat_response.choices[0].message.content)


InternalServerError: Error code: 500 - {'error': 'Error processing the request'}

In [32]:

output1 = client.embeddings.create(input = ["What is risk management in project management"],model=model_name)
embeding1 = output1.data[0].embedding

In [31]:
results1 = index.query(
    namespace="__default__",
    vector=embeding1,
    top_k=3,
    include_values=False,
    include_metadata=True
)

print(results1)

{'matches': [{'id': 'doc_10_0',
              'metadata': {'chunk_id': 0.0,
                           'source': 'SSM L6 Risk Management.txt',
                           'text': 'Filename: SSM L6 Risk Management.pptx '
                                   'Slide 1: Smart Systems Management L6 Risk '
                                   'Assessment and Mitigation DRAFT      1 '
                                   'Slide 2: Risk Assessment and Mitigation '
                                   'Risk Assessment and Mitigation are '
                                   'processes used to identify, evaluate, and '
                                   'address potential risks that could impact '
                                   'a project, organization, or system. 2 '
                                   'Slide 3: Risk Assessment and Mitigation '
                                   'Risk Assessment: is the process of '
                                   'identifying and evaluating risks. It '
           

In [ ]:
# Step 1: Extract metadata (texts) from the query results
retrieved_contexts1 = [match['metadata']['text'] for match in results1['matches']]

# Step 2: Prepare the prompt
query1 = "What is risk management in project management."

context1 = "\n".join(retrieved_contexts1)
final_prompt1 = f"""Use the following information to answer the question.

Context:
{context1}

Question:
{query1}

Answer:"""

# Step 3: Call the LLM using RunPod Chat Completion
chat_response1 = client.chat.completions.create(
    model=model_name,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": final_prompt1}
    ],
    temperature=0.2
)

# Step 4: Print the answer
print(chat_response1.choices[0].message.content)
